## Web Scraping

Like many other languages, Python includes libraries to perform web scraping. It is important to understand
how to scrape responsibly and the challenges that you may face. You might think:

> "If I can scrape it, I can use it."

This is NOT always true! Be careful!

Before we begin, consider the following two cases:
* [https://en.wikipedia.org/wiki/HiQ_Labs_v._LinkedIn](https://en.wikipedia.org/wiki/HiQ_Labs_v._LinkedIn)
* [https://en.wikipedia.org/wiki/Craigslist_Inc._v._3Taps_Inc](https://en.wikipedia.org/wiki/Craigslist_Inc._v._3Taps_Inc.)

Reminders:
 * <strong>Use APIs instead when possible</strong>
 * Only scrape data that is public (nothing behind a login), non-personal (avoid PII), and factual (avoid copyrighted data). It is often what you do with the data scraped that leads to issues. 
 * Talk to a lawyer first if you are planning on doing anything to make money with scraped data (possible copyright infringement).
 * Be respectful to server hosting pages that you are scraping (wait between requests). Too many requests too quickly can impact the service and result in a CFAA violation. 
 * Consider where you are (examples include California Consumer Privacy Act (CCPA) and rules that are different in different countries).
 * Consider term of service ([Example](https://help.nytimes.com/hc/en-us/articles/115014893428-Terms-of-Service)).

Common challenges to scraping:
 * Dynamic content structure -- difficult to scrape what you want
 * Page structure changes (similar to above)
 * You need to process and format data for data science projects
 * Dynamic content -- requires user interaction on page to load content (e.g. lazy loading) -- might require use of tools like Selenium
 * IP Bans and CAPTCHAs -- too many ill-timed requests may land you in this territory
 * Rate limits -- need to scrape responsibly to avoid  hitting limits
 * Honeypot -- your scraper follows a link meant only to be accessible by bots, then you are blocked
 * proxies are used to get around being blocked
 
You need to be very careful when choosing your source of data. APIs will instead provide a more predictable and reliable experience. Remember, if you are using web scraping irresponsibly, you will most likely be blocked from that site. You can check the robots.txt page of the site to determine crawl rates. Example: see [https://www.pct.edu/robots.txt](https://www.pct.edu/robots.txt)

The process of scraping involves typically **3 steps**:
1. Use an HTTP request to retrieve the content of the page.
2. Parse the resulting HTML content so that you can access nodes in the tree.
3. Navigate tree to extract what you need.

There are a few libraries/modules that you can import to help you scrape data. For example, urllib.request paired with the regular expression library (re) allows you to retrieve web page content and shift through the markup. This requires a lot of code. To make this an easier process, we are going to instead use the BeautifulSoup library.

### BeautifulSoup

The Python BeautifulSoup library allows us to more easily pull data from HTML and XML files. It is built on top of other Python libraries. To find more about what this library can do, refer to the documentation at [https://beautiful-soup-4.readthedocs.io/en/latest/#](https://beautiful-soup-4.readthedocs.io/en/latest/#).

In [5]:
# we will use the requests library to create a request for us (see step 1 earlier)
import requests 
# bring in the library -- note we are using version 4 as it works with Python 3
# for those with import issues: pip install beautifulsoup4
from bs4 import BeautifulSoup

# eventually we want all data in a DataFrame so...
import pandas as pd 
# needed for regular expression example below
import re 
# needed for delays below
from time import sleep

We will begin by getting the HTML we want.

In [ ]:
# the following site should be okay to scrape as it is designed to teach scraping in R
URL = "https://rvest.tidyverse.org/articles/starwars.html"
response = requests.get(URL)

Let's see what we've got using the "content" property of the request (just like we did with the APIs).

In [ ]:
response.content

If you are going to do multiple requests, you should respect rate limiting by putting delays between your requests. You can also randomize the wait times to mimic typical user behavior. Sometimes the robots.txt page of a site will provide you with the expected crawl delay.  

In [ ]:
sleep(3) # sleep 3 seconds

You can see how the above response.content would be pretty difficult to work with.  However, BeautifulSoup makes this easier. It can parse the content for us and build a tree that we can easily traverse.  First we need to create a BeautifulSoup object and pass it our content. Note that we can tell it what kind of content we have so that it parses the tree correctly.  Typically content types are:

* html.parser
* lxml
* lxml-xml or xml
* html5lib <-- slower but more accurate

See above [https://beautiful-soup-4.readthedocs.io/en/latest/#differences-between-parsers](https://beautiful-soup-4.readthedocs.io/en/latest/#differences-between-parsers) to understand difference between parsers.

We can use the prettify() function of BeautifulSoup to see the soup in human-readable form. 

In [ ]:
# Note: they call this "making the soup"
soup = BeautifulSoup(response.content, 'html.parser') 

# let's see this in a human-readable form


So we can use the soup object to specify an element(tag) name to access the first occurrence of a element. But this is pretty limiting. 

In [ ]:
# if a page just has one title, this makes sense
print(soup.title)
print(soup.title.text) # without the tag

# page has more than one h2, so this is less helpful
print(soup.h2)

It is a good idea to explore what you have by using the prettify() function above so that you can better understand what you need to extract. By exploring the HTML, we can see that each movie is in its own "section" tag and this section is followed by a h2 that contains the title of the movie. All information about the movie is nested in the "section" tag. Note using the inspector in Chrome is also pretty helpful here. 

For this example, we want to extract basic information about each movie: title, release date, director, description. We can use the **find_all()** function to find and retrieve all "section" tags. This will give us a list of that content. Note that the **find()** function just returns the first occurrence of an element. See [https://beautiful-soup-4.readthedocs.io/en/latest/#searching-the-tree](https://beautiful-soup-4.readthedocs.io/en/latest/#searching-the-tree) for more examples.

In [ ]:
# use find_all() to get all section tags


In addition to finding by tag, we can also find by id, text in enclosed in tags, attributes values, and class.

In [ ]:
# multiple elements
soup.find(["div", "section"])

# by id
soup.find(id = "main")

# by text
soup.find(string = "Last Updated")

# regular expression
soup.find(re.compile("^b")) # this example returns <body> content

# by attribute
soup.find(attrs = {"src":"logo.png"})

# by class
soup.find(class_ = "navbar")

# for those who love CSS, you can also search by CSS selector
soup.select(".navbar > a")

Now that we've isolated what we want, we can just traverse the tree using tag names. It is super easy.

In [ ]:
# the h2 contains the title
section_list[0].h2

If there are no child tags (no other tags inside the h2), then we can use the .string or .contents properties to obtain the text.

In [ ]:
# get just the text by adding the .string or .contents to the code above


In [ ]:
# the first paragraph contains the release date
# note that this gives us access to the first occurrence of the p tag in the section
section_list[0].p

In [ ]:
# the span contains the director
section_list[0].span

In [ ]:
# and finally, the description (paragraphs in the div)


The above output shows us that some tags will have children tags (tags nested inside them). You can access this content the same way as above.

In [ ]:
print(section_list[0].div.p)

In [ ]:
# iterate through to see them
for p in section_list[0].div.children:
    print(p)

In [ ]:
# the above lets us see the content. We can send it to the list() function to store it
description = list(section_list[0].div.children)
description

Note that can also retrieve tags with a given attribute by using find_all with additional attributes.

In [ ]:
soup.find_all("span", class_="director") # class is a keyword in Python, so we have to instead use class_ here

There is much more we can do, but let's look at how to put this data into a DataFrame for easier analysis.  Pandas has a [read_html()](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html) function that takes an HTML string and uses a built-in parser to convert the data. This provides a quick way to generate a DataFrame. 

Like we did with APIs, you might want to first select what HTML you want to convert instead of attempting to generate the DataFrame from an entire page. Note that the **read_html()** function has a variety of parameters that also help you select exactly the data you want to use to create the DataFrame. **By default, read_html() looks for a table in the HTML.** It also actually returns a list, not just a DataFame. But in that list you'll find your DataFrame.

In [ ]:
# Here's an example when a page contains a table
# url to use https://en.wikipedia.org/wiki/Usage_share_of_web_browsers#W3Counter_(May_2007_to_December_2022)
# make request
headers = { "User-agent": "MyPythonScript/1.0" }
response = requests.get("https://en.wikipedia.org/wiki/Usage_share_of_web_browsers#W3Counter_(May_2007_to_December_2022)", headers=headers)

# make the soup using html.parser as the parser of the result content


In [ ]:
# Let's peek at content with prettify()


In [ ]:
# Get all tables - use find_all to find all tables on this page - and store as list


In [ ]:
# I only really want one of these... the 7th table on the page
# something like data = table[6]

In [ ]:
# We can now use the read_html to get the data
browser_stats = pd.read_html(str(data))
browser_stats # is a list

In [ ]:
# first item in list is our DataFame...let's just get that
df = browser_stats[0] 
df

Let's finish our Star Wars example by putting the data in a DataFrame. 

In [ ]:
titles = []
release_dates = []
directors = []
descriptions = []

for section in section_list:
    titles.append(section.h2.string)

for section in section_list:
    release_dates.append(section.p.string)
    
for section in section_list:
    directors.append(section.span.string)
    
for section in section_list:
    descriptions.append([x.string for x in section.div.children])

dict = {"title" : titles, "release_date" : release_dates, "director" : directors, "descriptions" : descriptions}

df = pd.DataFrame(dict)
df

In [ ]:
# alternate way
data = []

# iterate through results
for section in section_list:
    row = []
    row.append(section.h2.string)
    row.append(section.p.string)
    row.append(section.span.string)
    row.append([x.string for x in section.div.children]) # quick list comprehension
    data.append(row)

movies_df = pd.DataFrame(data, columns=["title", "release_date", "director", "description"])
movies_df


In [ ]:
# Clean up data a bit
movies_df = movies_df.replace('\n', '', regex=True)
movies_df = movies_df.replace('Released: ', '', regex=True)
movies_df

In [ ]:
# let's explore what we have
movies_df.info()

In [ ]:
# just checking more stuff out
movies_df.description

## Questions

1. Pull data from: https://www.wikidata.org/wiki/Wikidata:WikiProject_Music/Triple_J_Hottest_100_Music_Chart

2. Imagine you just wanted the list of performers. Determine the selector you would use to obtain a list of the performers

3. Place all table data in a dataframe

5. Use the [groupby function](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) of the pandas library to group records by performer and determine which performers have over 3 hits on the chart.

In [ ]:
# 1 
response = requests.get("https://www.wikidata.org/wiki/Wikidata:WikiProject_Music/Triple_J_Hottest_100_Music_Chart", headers={"User-agent":"PythonScript/1.0"})